[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLModel, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)

# A Small Service &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup and the project its worked examples wrote: the five files,
the migration that builds the schema, and the six tests passing. Run it first. The tasks add to the
same project in order, and the last cell removes the scratch folder.


In [1]:
import os
import re
import shlex
import shutil
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

for package, pin in (("sqlmodel", "sqlmodel==0.0.42"), ("fastapi", "fastapi==0.141.1"),
                     ("httpx", "httpx==0.28.1"), ("alembic", "alembic==1.20.0"), ("pytest", "pytest==8.4.2")):
    try:
        version(package)
    except PackageNotFoundError:                                    # install what this runtime is missing
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore", pin],
                       check=True)

import sqlmodel

os.environ["NO_COLOR"] = "1"                                        # no terminal codes in what a tool prints
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"                         # no stale compiled copy of a file just rewritten

SERVICE = Path("scratch/service")


def shell(*arguments, module=True):
    """Run a command in the service folder and print what it said, with the folder's own path taken out."""
    command = [sys.executable, "-m", *arguments] if module else [sys.executable, *arguments]
    done = subprocess.run(command, cwd=SERVICE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                          env={**os.environ, "COLUMNS": "80", "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1",
                               "PYTHONNODEBUGRANGES": "1", "PYTHONUNBUFFERED": "1",
                               "PYTHONWARNINGS": "ignore"})         # a warning names a file inside a library
    report = done.stdout.replace(f"{SERVICE.resolve()}{os.sep}", "")
    report = re.sub(r"\S*/_pytest/", "_pytest/", report)            # the path to pytest's own files
    report = re.sub(r"0x[0-9a-f]+", "0x...", report)                # the memory addresses of objects
    report = re.sub(r" in \d+\.\d+s\b", "", report)                 # the time a run took
    print("$", shlex.join(["python", "-m", *arguments] if module else ["python", *arguments]))
    for line in report.rstrip().splitlines():
        if not any(noise in line for noise in ("Context impl", "Will assume", "Please edit",
                                               "setting up autogenerate plugin")):
            print("   ", line)


def failures(*arguments):
    """Run pytest and print only what failed, which is what a reader needs from a report with a failure in it."""
    done = subprocess.run([sys.executable, "-m", "pytest", "--no-header", "-q", *arguments], cwd=SERVICE,
                          capture_output=True, text=True,
                          env={**os.environ, "COLUMNS": "80", "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1",
                               "PYTHONNODEBUGRANGES": "1"})
    report = re.sub(r" in \d+\.\d+s\b", "", done.stdout + done.stderr)
    for line in report.splitlines():
        if line.startswith(("E ", "FAILED", "ERROR")) or re.match(r"\d+ (passed|failed)", line.strip()):
            print(line.replace(f"{SERVICE.resolve()}{os.sep}", "")[:100])


def edit(path, old, new):
    """Change one piece of a file, and fail rather than silently do nothing."""
    text = path.read_text()
    assert text.count(old) == 1, f"{path.name}: found {text.count(old)} of {old!r}"
    path.write_text(text.replace(old, new))

shutil.rmtree("scratch", ignore_errors=True)                        # a rerun starts from no project at all
SERVICE.mkdir(parents=True)

print("sqlmodel", sqlmodel.__version__, "| fastapi", version("fastapi"), "| alembic", version("alembic"),
      "| pytest", version("pytest"))

PROJECT = {
    "models.py": r'''from sqlmodel import Field, Relationship, SQLModel


class Team(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, unique=True, max_length=50)
    headquarters: str = Field(max_length=60)

    heroes: list["Hero"] = Relationship(back_populates="team", cascade_delete=True)


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, unique=True, max_length=50)
    secret_name: str = Field(max_length=60)                 # never leaves the service
    age: int | None = Field(default=None, index=True)
    team_id: int | None = Field(default=None, foreign_key="team.id", ondelete="CASCADE")

    team: Team | None = Relationship(back_populates="heroes")


class HeroCreate(SQLModel):
    name: str = Field(max_length=50)
    secret_name: str = Field(max_length=60)
    age: int | None = None
    team_id: int | None = None


class HeroUpdate(SQLModel):
    name: str | None = None
    secret_name: str | None = None
    age: int | None = None
    team_id: int | None = None


class TeamPublic(SQLModel):
    id: int
    name: str
    headquarters: str


class HeroPublic(SQLModel):
    id: int
    name: str
    age: int | None = None


class HeroWithTeam(HeroPublic):
    team: TeamPublic | None = None
''',
    "database.py": r'''import os

from sqlalchemy import event
from sqlalchemy.engine import Engine
from sqlmodel import Session, create_engine

DATABASE = os.environ.get("HEROES_DATABASE", "heroes.db")
engine = create_engine(f"sqlite:///{DATABASE}")


@event.listens_for(Engine, "connect")
def keep_foreign_keys(connection, record):
    """SQLite checks a foreign key only where this pragma is on, and it is off on every new connection."""
    cursor = connection.cursor()
    cursor.execute("PRAGMA foreign_keys=ON")
    cursor.close()


def get_session():
    """One session for one request, closed when the request is finished."""
    with Session(engine) as session:
        yield session
''',
    "main.py": r'''from fastapi import Depends, FastAPI, HTTPException
from sqlalchemy.exc import IntegrityError
from sqlalchemy.orm import selectinload
from sqlmodel import Session, select

from database import get_session
from models import Hero, HeroCreate, HeroPublic, HeroUpdate, HeroWithTeam, Team

app = FastAPI(title="Heroes")


@app.post("/heroes", response_model=HeroPublic, status_code=201)
def create_hero(arriving: HeroCreate, session: Session = Depends(get_session)):
    hero = Hero.model_validate(arriving)
    session.add(hero)
    try:
        session.commit()
    except IntegrityError:
        session.rollback()
        raise HTTPException(status_code=409, detail=f"{arriving.name} is taken")
    session.refresh(hero)
    return hero


@app.get("/heroes", response_model=list[HeroWithTeam])
def list_heroes(page: int = 1, per_page: int = 5, session: Session = Depends(get_session)):
    listed = (select(Hero).order_by(Hero.name)
              .offset((page - 1) * per_page).limit(per_page)
              .options(selectinload(Hero.team)))
    return session.exec(listed).all()


@app.get("/heroes/{hero_id}", response_model=HeroWithTeam)
def read_hero(hero_id: int, session: Session = Depends(get_session)):
    hero = session.exec(select(Hero).where(Hero.id == hero_id)
                        .options(selectinload(Hero.team))).one_or_none()
    if hero is None:
        raise HTTPException(status_code=404, detail="no hero with that id")
    return hero


@app.patch("/heroes/{hero_id}", response_model=HeroPublic)
def change_hero(hero_id: int, change: HeroUpdate, session: Session = Depends(get_session)):
    hero = session.get(Hero, hero_id)
    if hero is None:
        raise HTTPException(status_code=404, detail="no hero with that id")
    hero.sqlmodel_update(change.model_dump(exclude_unset=True))      # only what the caller sent
    session.add(hero)
    session.commit()
    session.refresh(hero)
    return hero


@app.delete("/teams/{team_id}", status_code=204)
def delete_team(team_id: int, session: Session = Depends(get_session)):
    team = session.get(Team, team_id)
    if team is None:
        raise HTTPException(status_code=404, detail="no team with that id")
    session.delete(team)                                             # its heroes go with it
    session.commit()
''',
    "conftest.py": r'''import pytest
from alembic import command
from alembic.config import Config
from fastapi.testclient import TestClient
from sqlmodel import Session, create_engine

from database import get_session
from main import app


@pytest.fixture(name="engine")
def engine_fixture(tmp_path):
    """A database of this run's own, built by the migrations rather than by create_all."""
    database = tmp_path / "test.db"
    config = Config("alembic.ini")
    config.set_main_option("sqlalchemy.url", f"sqlite:///{database}")
    command.upgrade(config, "head")
    made = create_engine(f"sqlite:///{database}")
    yield made
    made.dispose()


@pytest.fixture(name="session")
def session_fixture(engine):
    """A session on that database, for a test that talks to the models directly."""
    with Session(engine) as session:
        yield session


@pytest.fixture(name="client")
def client_fixture(engine):
    """A client whose requests use that database, and an application otherwise untouched."""
    def override():
        with Session(engine) as session:
            yield session

    app.dependency_overrides[get_session] = override
    yield TestClient(app)
    app.dependency_overrides.clear()
''',
    "test_service.py": r'''from sqlmodel import select

from models import Hero, Team


def test_a_hero_is_created_and_read_back(client):
    made = client.post("/heroes", json={"name": "Deadpond", "secret_name": "Dive Wilson", "age": 30})
    assert made.status_code == 201
    hero_id = made.json()["id"]

    read = client.get(f"/heroes/{hero_id}")
    assert read.status_code == 200
    assert read.json()["name"] == "Deadpond"


def test_a_missing_hero_is_a_404(client):
    assert client.get("/heroes/999").status_code == 404


def test_a_repeated_name_is_a_409(client):
    client.post("/heroes", json={"name": "Deadpond", "secret_name": "Dive Wilson"})
    again = client.post("/heroes", json={"name": "Deadpond", "secret_name": "Someone Else"})
    assert again.status_code == 409


def test_a_body_that_is_wrong_is_a_422(client):
    refused = client.post("/heroes", json={"name": "Ghost", "secret_name": "Ana", "age": "old"})
    assert refused.status_code == 422
    assert refused.json()["detail"][0]["loc"] == ["body", "age"]


def test_the_list_is_paged(client):
    for number in range(7):
        client.post("/heroes", json={"name": f"Hero {number}", "secret_name": f"Secret {number}"})
    first = client.get("/heroes", params={"page": 1, "per_page": 5}).json()
    second = client.get("/heroes", params={"page": 2, "per_page": 5}).json()
    assert len(first) == 5
    assert len(second) == 2


def test_deleting_a_team_takes_its_heroes(client, session):
    session.add(Team(name="Preventers", headquarters="Sharp Tower"))
    session.commit()
    client.post("/heroes", json={"name": "Rusty-Man", "secret_name": "Tommy Sharp", "team_id": 1})

    assert client.delete("/teams/1").status_code == 204
    assert session.exec(select(Hero)).all() == []
''',
    "pytest.ini": r'''[pytest]
filterwarnings = ignore
''',
}
for name, text in PROJECT.items():
    (SERVICE / name).write_text(text)


shell("alembic", "init", "migrations")
ini = SERVICE / "alembic.ini"
ini.write_text(re.sub(r"^sqlalchemy\.url = .*$", "sqlalchemy.url = sqlite:///heroes.db",
                      ini.read_text(), flags=re.M))
edit(SERVICE / "migrations" / "env.py", "target_metadata = None",
     'import sys\n\nsys.path.insert(0, ".")\nfrom models import SQLModel\n\ntarget_metadata = SQLModel.metadata')
edit(SERVICE / "migrations" / "script.py.mako", "import sqlalchemy as sa",
     "import sqlalchemy as sa\nimport sqlmodel")

shell("alembic", "revision", "--autogenerate", "-m", "heroes and teams", "--rev-id", "0001")
shell("alembic", "upgrade", "head")
shell("pytest", "--no-header", "-q")


sqlmodel 0.0.42 | fastapi 0.141.1 | alembic 1.20.0 | pytest 8.4.2
$ python -m alembic init migrations
    Creating directory migrations ...  done
    Creating directory migrations/versions ...  done
    Generating migrations/script.py.mako ...  done
    Generating migrations/env.py ...  done
    Generating migrations/README ...  done
    Generating alembic.ini ...  done
$ python -m alembic revision --autogenerate -m 'heroes and teams' --rev-id 0001
    INFO  [alembic.autogenerate.compare.tables] Detected added table 'team'
    INFO  [alembic.autogenerate.compare.constraints] Detected added index 'ix_team_name' on '('name',)'
    INFO  [alembic.autogenerate.compare.tables] Detected added table 'hero'
    INFO  [alembic.autogenerate.compare.constraints] Detected added index 'ix_hero_age' on '('age',)'
    INFO  [alembic.autogenerate.compare.constraints] Detected added index 'ix_hero_name' on '('name',)'
    Generating migrations/versions/0001_heroes_and_teams.py ...  done
$ python -m ale

**1.** A team can be created, and a name cannot be taken twice.


In [2]:
edit(SERVICE / "models.py", "class HeroCreate(SQLModel):",
     "class TeamCreate(SQLModel):\n"
     "    name: str = Field(max_length=50)\n"
     "    headquarters: str = Field(max_length=60)\n\n\n"
     "class HeroCreate(SQLModel):")
edit(SERVICE / "main.py", "from models import Hero, HeroCreate, HeroPublic, HeroUpdate, HeroWithTeam, Team",
     "from models import Hero, HeroCreate, HeroPublic, HeroUpdate, HeroWithTeam, Team, TeamCreate, TeamPublic")
edit(SERVICE / "main.py", '@app.get("/heroes", response_model=list[HeroWithTeam])',
     '@app.post("/teams", response_model=TeamPublic, status_code=201)\n'
     'def create_team(arriving: TeamCreate, session: Session = Depends(get_session)):\n'
     '    team = Team.model_validate(arriving)\n'
     '    session.add(team)\n'
     '    try:\n'
     '        session.commit()\n'
     '    except IntegrityError:\n'
     '        session.rollback()\n'
     '        raise HTTPException(status_code=409, detail=f"{arriving.name} is taken")\n'
     '    session.refresh(team)\n'
     '    return team\n\n\n'
     '@app.get("/heroes", response_model=list[HeroWithTeam])')

(SERVICE / "test_teams.py").write_text("""
def test_a_team_is_created(client):
    made = client.post("/teams", json={"name": "Preventers", "headquarters": "Sharp Tower"})
    assert made.status_code == 201
    assert made.json()["name"] == "Preventers"


def test_a_repeated_team_name_is_a_409(client):
    client.post("/teams", json={"name": "Preventers", "headquarters": "Sharp Tower"})
    again = client.post("/teams", json={"name": "Preventers", "headquarters": "Somewhere Else"})
    assert again.status_code == 409
""")

shell("pytest", "--no-header", "-q", "test_teams.py")


$ python -m pytest --no-header -q test_teams.py
    ..                                                                       [100%]
    2 passed


The same shape as the hero route, with its own create model, and the unique name in the models is
what the 409 rests on.


**2.** Teams with their heroes.


In [3]:
edit(SERVICE / "models.py", "class HeroWithTeam(HeroPublic):",
     "class TeamWithHeroes(TeamPublic):\n    heroes: list[HeroPublic] = []\n\n\n"
     "class HeroWithTeam(HeroPublic):")
edit(SERVICE / "main.py", "HeroWithTeam, Team, TeamCreate, TeamPublic",
     "HeroWithTeam, Team, TeamCreate,\n                    TeamPublic, TeamWithHeroes)")
edit(SERVICE / "main.py", "from models import Hero", "from models import (Hero")
edit(SERVICE / "main.py", '@app.get("/heroes/{hero_id}", response_model=HeroWithTeam)',
     '@app.get("/teams", response_model=list[TeamWithHeroes])\n'
     'def list_teams(session: Session = Depends(get_session)):\n'
     '    return session.exec(select(Team).order_by(Team.name)\n'
     '                        .options(selectinload(Team.heroes))).all()\n\n\n'
     '@app.get("/heroes/{hero_id}", response_model=HeroWithTeam)')

(SERVICE / "test_team_list.py").write_text("""
def test_a_team_carries_its_heroes(client):
    client.post("/teams", json={"name": "Preventers", "headquarters": "Sharp Tower"})
    client.post("/heroes", json={"name": "Rusty-Man", "secret_name": "Tommy Sharp", "team_id": 1})

    listed = client.get("/teams").json()
    assert [hero["name"] for hero in listed[0]["heroes"]] == ["Rusty-Man"]
""")

shell("pytest", "--no-header", "-q", "test_team_list.py")


$ python -m pytest --no-header -q test_team_list.py
    .                                                                        [100%]
    1 passed


`TeamWithHeroes` carries `HeroPublic`, which carries no team, so the nesting stops after one level
and there is no loop for FastAPI to detect. `selectinload` keeps the list two statements.


**3.** A motto, and the revision that adds it.


In [4]:
MOTTO = "    motto: str | None = Field(default=None, max_length=80)"
edit(SERVICE / "models.py", "    headquarters: str = Field(max_length=60)\n\n    heroes:",
     f"    headquarters: str = Field(max_length=60)\n{MOTTO}\n\n    heroes:")

shell("alembic", "revision", "--autogenerate", "-m", "a motto", "--rev-id", "0002")
shell("alembic", "upgrade", "head")
shell("alembic", "check")


$ python -m alembic revision --autogenerate -m 'a motto' --rev-id 0002
    INFO  [alembic.autogenerate.compare.tables] Detected added column 'team.motto'
    Generating migrations/versions/0002_a_motto.py ...  done
$ python -m alembic upgrade head
    INFO  [alembic.runtime.migration] Running upgrade 0001 -> 0002, a motto
$ python -m alembic check
    No new upgrade operations detected.


The revision adds one column, and `alembic check` finds nothing left to do: the models and the
migrations describe the same schema again.


**4.** A change that leaves the rest alone.


In [5]:
(SERVICE / "test_patch.py").write_text("""
def test_a_change_of_age_leaves_the_name(client):
    client.post("/heroes", json={"name": "Deadpond", "secret_name": "Dive Wilson", "age": 30})
    changed = client.patch("/heroes/1", json={"age": 31})

    assert changed.status_code == 200
    assert changed.json() == {"id": 1, "name": "Deadpond", "age": 31}
""")

shell("pytest", "--no-header", "-q", "test_patch.py")


$ python -m pytest --no-header -q test_patch.py
    .                                                                        [100%]
    1 passed


The answer is the whole public model, so the test can compare it in one line: the age changed and
the name did not.


**5.** A hero may not be put on a team that is not there.


In [6]:
(SERVICE / "test_missing_team.py").write_text("""
def test_a_hero_needs_a_team_that_exists(client):
    refused = client.post("/heroes", json={"name": "Ghost Girl", "secret_name": "Ana Vega",
                                           "team_id": 999})
    assert refused.status_code == 409
""")

shell("pytest", "--no-header", "-q", "test_missing_team.py")
# It passes already: database.py turns the foreign key pragma on for every connection, so SQLite
# refuses the row, and the create route catches IntegrityError and answers 409.


$ python -m pytest --no-header -q test_missing_team.py
    .                                                                        [100%]
    1 passed


**6.** The whole suite.


In [7]:
shell("pytest", "--no-header", "-q")


$ python -m pytest --no-header -q
    ...........                                                              [100%]
    11 passed


Every test in the project, each on a database the migrations built for it and threw away
afterwards.

Last, this cell removes the scratch folder with the service in it:


In [8]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [A Small Service](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/14-a-small-service.ipynb)  &nbsp;&middot;&nbsp;  [SQLModel, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)
